In [13]:
import os
from dotenv import load_dotenv
from opendartreader import OpenDartReader

# .env 파일의 환경 변수를 불러옵니다
load_dotenv()

# 환경 변수에서 키 값을 꺼내옵니다
api_key = os.environ.get('DART_API_KEY')

dart = OpenDartReader(api_key)
print("✅ DART API 연결 완료")

✅ DART API 연결 완료


In [14]:
import FinanceDataReader as fdr # 한국 및 해외 주식 가격, 종목 리스트
import pandas as pd

# API 키 설정
api_key = '5c89cd192f89c298966614caf80a7d4ef27fd9f0'
dart = OpenDartReader(api_key)

# DART에 등록된 모든 기업의 고유번호 목록 가져오기
# (XML을 다운받아 DataFrame으로 반환)
dart_corp_list = dart.corp_codes
print(f"DART 등록 기업 수: {len(dart_corp_list)}")

# KOSPI 종목 리스트 가져오기 (시가총액 순으로 정렬되어 있음)
# '2026'년 기준으로 가장 최근 KOSPI 데이터
kospi_list = fdr.StockListing('KOSPI')

# KOSPI 시가총액 상위 50개 기업 추출
top_50_kospi = kospi_list.head(50)[['Code', 'Name', 'Marcap']]
top_50_kospi = top_50_kospi.rename(columns={'Code': 'stock_code', 'Name': 'corp_name'})

# DART 고유번호와 KOSPI 상위 50개 기업 매핑 (Merge)
# DART 데이터의 'stock_code'와 KOSPI 데이터의 'stock_code'를 기준으로 조인
mapped_df = pd.merge(top_50_kospi, dart_corp_list, how='left', on='stock_code')

# 확인
display(mapped_df.head())

DART 등록 기업 수: 118767


,stock_code,corp_name_x,Marcap,corp_code,corp_name_y,corp_eng_name,modify_date
0,005930,삼성전자,1528801855992000,00126380,삼성전자,"SAMSUNG ELECTRONICS CO,.LTD",20251201
1,000660,SK하이닉스,1233071112120000,00164779,SK하이닉스,SK hynix Inc.,20240328
2,005935,삼성전자우,157906652750400,NaN,NaN,NaN,NaN
3,402340,SK스퀘어,139575589884000,01596425,SK스퀘어,"SK Square Co., Ltd.",20260326
4,009150,삼성전기,99342615680000,00126371,삼성전기,"SAMSUNG ELECTRO-MECHANICS CO.,LTD",20230102


In [15]:
# 우선주(corp_code가 NaN인 데이터) 제거
clean_mapped_df = mapped_df.dropna(subset=['corp_code']).copy()

# 불필요한 컬럼 정리 (corp_name_y 등 제거하고 이름 통일)
clean_mapped_df = clean_mapped_df[['stock_code', 'corp_code', 'corp_name_x', 'Marcap']]
clean_mapped_df = clean_mapped_df.rename(columns={'corp_name_x': 'corp_name'})

# KOSPI 상위 50개 매핑 데이터 저장 (향후 DB 적재용)
os.makedirs('../data', exist_ok=True) # data 폴더가 없다면 생성
clean_mapped_df.to_csv('../data/kospi_top50_mapping.csv', index=False, encoding='utf-8-sig')

display(clean_mapped_df.head())

,stock_code,corp_code,corp_name,Marcap
0,005930,00126380,삼성전자,1528801855992000
1,000660,00164779,SK하이닉스,1233071112120000
3,402340,01596425,SK스퀘어,139575589884000
4,009150,00126371,삼성전기,99342615680000
5,005380,00164742,현대차,83541168528000
